In [1]:
import pandas as pd

In [19]:
df_movies = pd.read_csv('../data/df_final.csv')
df_tsne = pd.read_csv('../data/df_tsne.csv')

X_reduced = df_tsne[['Dim1', 'Dim2']]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# matrice de similarité
similarity_matrix = cosine_similarity(X_reduced)

# tester : similarité du premier film avec tous les autres
similarity_matrix[0]

array([ 1.        ,  0.99936437,  0.99868076, ..., -0.77325251,
       -0.71017153, -0.30995832], shape=(9742,))

In [22]:
def recommend_movies(title, df, similarity_matrix, top_n=3):

    # Vérifier que la colonne title existe
    if 'title' not in df.columns:
        return "Erreur : la colonne 'title' n'existe pas dans le dataframe."

    # Recherche insensible à la casse
    matches = df[df['title'].str.lower() == title.lower()]

    if matches.empty:
        # Suggestion de titres proches
        suggestions = df[df['title'].str.contains(title, case=False, na=False)]['title'].head(5).values
        
        if len(suggestions) > 0:
            return f"Film non trouvé. Suggestions possibles : {list(suggestions)}"
        else:
            return "Film non trouvé dans la base de données."

    idx = matches.index[0]

    sim_scores = list(enumerate(similarity_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n+1]

    recommended_titles = df.iloc[
        [i for i, _ in sim_scores]
    ]['title'].values

    return recommended_titles


recommend_movies('Toy Story (1995)', df_movies, similarity_matrix, top_n=3)

<StringArray>
['Life Aquatic with Steve Zissou, The (2004)',
                  'That Thing You Do! (1996)',
                   'Pineapple Express (2008)']
Length: 3, dtype: str

In [43]:
df_pca = pd.read_csv('../data/df_pca.csv')

X_features = df_pca[['PC1', 'PC2']]

In [44]:
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
import numpy as np

def recommend_movies_multiple(titles, df, X_features, top_n=5):
    """
    titles : liste de titres sélectionnés
    df : dataframe contenant la colonne 'title'
    X_features : matrice des features utilisées pour calculer similarity
    top_n : nombre de recommandations
    """
    # Vérification des titres
    valid_titles = [t for t in titles if t in df['title'].values]
    
    if len(valid_titles) == 0:
        return "Aucun film valide sélectionné."
    
    # Récupérer les indices des films
    indices = [df[df['title'] == t].index[0] for t in valid_titles]
    
    # Moyenne des vecteurs
    mean_vector = X_features[indices].mean(axis=0).reshape(1, -1)
    
    # Calculer similarité cosinus
    sim = cosine_similarity(mean_vector, X_features)[0]
    
    # Trier et récupérer top_n
    sorted_indices = np.argsort(sim)[::-1]
    
    # Exclure les films déjà sélectionnés
    sorted_indices = [i for i in sorted_indices if i not in indices]
    
    recommended_titles = df.iloc[sorted_indices[:top_n]]['title'].values
    
    return recommended_titles

# test 
recommend_movies_multiple(['Toy Story (1995)', 'Jumanji (1995)'], df_movies, X_features.values, top_n=5)

<StringArray>
['The Lego Batman Movie (2017)',        'Public Enemies (2009)',
                  'Wolf (1994)',             'Peter Pan (1953)',
               'Aladdin (1992)']
Length: 5, dtype: str

In [ ]:
# --- Liste des films sélectionnés ---
selected_movies = []

# --- Barre de recherche avec auto-complétion ---
search_bar = widgets.Combobox(
    placeholder='Tape un titre...',
    options=list(df_movies['title']),
    description='Film :',
    ensure_option=False,
    layout=widgets.Layout(width='500px')
)

# --- Bouton Ajouter ---
add_button = widgets.Button(description="Add", button_style='success')


# --- Zone affichage liste ---
multi_select_box = widgets.SelectMultiple(
    options=selected_movies,
    description='Sélection :',
    disabled=False,
    layout=widgets.Layout(width='500px')
)

# --- Bouton Remove  ---
remove_button = widgets.Button(description="Remove", button_style='danger')

# --- Bouton Recommandation ---
recommend_button = widgets.Button(description="Lancer recommandation", button_style='info')

# --- Output ---
output = widgets.Output()


def add_movie(b):
    title = search_bar.value.strip()
    
    if title in df_movies['title'].values:
        if title not in selected_movies:
            selected_movies.append(title)
            multi_select_box.options = selected_movies
        else:
            with output:
                clear_output()
                print("Film déjà ajouté.")
    else:
        with output:
            clear_output()
            print("Film non trouvé.")

def remove_movie(b):
    title = multi_select_box.value
    if title:
        selected_movies.remove(title[0])
        multi_select_box.options = selected_movies
            
add_button.on_click(add_movie)
remove_button.on_click(remove_movie)

def run_recommendation_multiple(b):
    with output:
        clear_output()
        
        selected = list(multi_select_box.options)
        if len(selected) == 0:
            print("Aucun film sélectionné.")
            return
        
        result = recommend_movies_multiple(
            selected,
            df_movies,
            X_features.values,
            top_n=5
        )
        
        if isinstance(result, str):
            print(result)
        else:
            print("Recommandations basées sur :", selected)
            display(df_movies[df_movies['title'].isin(result)])

recommend_button.on_click(run_recommendation_multiple)


ui = widgets.VBox([
    widgets.HBox([search_bar, add_button]),
    multi_select_box,
    remove_button,
    recommend_button,
    output
])

display(ui)